# 045-EDIT EDIT DISTANCE

## 背景
- HAMM ではハミング距離を使うことで進化的距離を求めた
- しかし、実践では、インデル変異により比較したい配列の長さが違うことがある
- 挿入も、欠失も、置換も同じく一つの変化として考え、一方の配列をもう一方にするために最小の操作数を考える必要がある
- これはまさにレーベンシュタイン距離(edit distance)である

## 問題設定
- ある二つの配列s, tの edit distance は、s を tにするための最小操作回数である
- 操作とは、一つの文字の置換、挿入、欠失のことである 
- 2 文字の挿入、欠失は 2 回の操作である

- Given : Fasta format で与えられる 2つのタンパク質配列 s, t (長さ 1000 以下)
- Return : edit distance

## 解法
- DP を使う
- (空文字 + s の各文字) × (空文字 + t の各文字) の dp テーブルを作る
- s の k 文字目までと t の j 文字目までを一致させるための最小操作数でテーブルを埋めていく

In [ ]:
fasta =""">Rosalind_39
PLEASANTLY
>Rosalind_11
MEANLY
"""

In [ ]:
# read fasta
def read_fasta(fasta: str) -> dict[str, str]:
    sequences = {}
    header = None
    seq = []
    fasta = fasta.splitlines()
    for line in fasta:
        if line.startswith('>'):
            if header is not None:
                sequences[header] = ''.join(seq)
            header = line[1:]  # Remove '>'
            seq = []
        else:
            seq.append(line)
    if header is not None:
        sequences[header] = ''.join(seq)  # Add the last sequence
    return sequences

In [ ]:
import numpy as np
def levanstein_distance(s1: str, s2: str) -> int:
    # 空文字 + 文字列の長さ の テーブルを作成
    dp_table = np.zeros((len(s1) + 1, len(s2) + 1), dtype=int)

    # 空文字からの距離を初期化する
    for i in range(len(s1) + 1):
        dp_table[i][0] = i
    for j in range(len(s2) + 1):
        dp_table[0][j] = j
    
    # テーブルを埋める
    # 上のマス + 1, 左のマス + 1, 左上のマス + (文字が違う場合は + 1) のうち最小の値を入れる
    # 右上から左下に向かって埋めていく
    for i in range(1, len(s1) + 1):
        for j in range(1, len(s2) + 1):
            
            up = dp_table[i][j - 1] + 1 # 上のマス + 1, 挿入
            left = dp_table[i - 1][j] + 1 # 左のマス + 1, 削除

            # 文字が同じなら置換は必要ない
            if s1[i - 1] == s2[j - 1]:
                left_up = dp_table[i - 1][j - 1] 
            else:
                left_up = dp_table[i - 1][j - 1] + 1
            
            dp_table[i][j] = min(up, left, left_up)
    
    return dp_table[-1][-1]

In [ ]:
fasta = read_fasta(fasta)
s1, s2 = fasta.values()
print(levanstein_distance(s1, s2))